In [ ]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt

I0000 00:00:1785304367.309368  344355 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785304367.825373  344355 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785304372.157538  344355 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
metadata_path = "../dataset/HAM10000_metadata.csv"
df = pd.read_csv(metadata_path)
df.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [3]:
image_paths = []
image_folders = [
    "../dataset/ham10000_images_part_1",
    "../dataset/ham10000_images_part_2"
]
for image_id in df["image_id"]:
    image_path = None
    for folder in image_folders:
        path = os.path.join(folder,image_id + ".jpg")
        if os.path.exists(path):
            image_path = path
            break
    image_paths.append(image_path)
df["image_path"] = image_paths

In [4]:
df["image_path"].isnull().sum()

np.int64(0)

Encode the "dx" target into labelencoder()

In [5]:
encoder = LabelEncoder()
df["label"] = encoder.fit_transform(df["dx"])

In [ ]:
class_mapping = dict(zip(encoder.classes_,encoder.transform(encoder.classes_)))
class_mapping

{'akiec': np.int64(0),
 'bcc': np.int64(1),
 'bkl': np.int64(2),
 'df': np.int64(3),
 'mel': np.int64(4),
 'nv': np.int64(5),
 'vasc': np.int64(6)}

Split the dataset into train,validation and test for triain the datasets.

In [ ]:
train_data, temp_data = train_test_split(df,test_size=0.30,random_state=42,stratify=df["label"])

In [ ]:
val_data, test_data = train_test_split(temp_data,test_size=0.50,random_state=42,stratify=temp_data["label"])

In [ ]:
print(train_data.shape,val_data.shape,test_data.shape)

(7010, 9) (1502, 9) (1503, 9)


Export the splited datasets

In [ ]:
train_data.to_csv("../dataset/train_metadata.csv",index=False)
val_data.to_csv("../dataset/val_metadata.csv",index=False)
test_data.to_csv("../dataset/test_metadata.csv",index=False)

Image Resize and Normaliation

In [ ]:
IMAGE_SIZE = 224
def preprocess_image(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(IMAGE_SIZE, IMAGE_SIZE))
    image = image / 255.0
    return image.astype(np.float32)

In [12]:
sample_image = preprocess_image(
    train_data.iloc[0]["image_path"]
)


sample_image.shape

(224, 224, 3)

In [ ]:
def create_dataset(data):
    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths,labels))
    def load_image(path,label):
        image = tf.numpy_function(preprocess_image,[path],tf.float32)
        image.set_shape((224,224,3))
        return image,label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [ ]:
train_dataset = create_dataset(train_data)
val_dataset = create_dataset(val_data)
test_dataset = create_dataset(test_data)

E0000 00:00:1785304392.969693  344355 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [ ]:
data_augmentation = tf.keras.Sequential([

    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

In [ ]:
def apply_augmentation(image,label):
    image = data_augmentation(image)
    return image,label
train_dataset = train_dataset.map(apply_augmentation,num_parallel_calls=tf.data.AUTOTUNE)

In [ ]:
BATCH_SIZE = 32
train_dataset = (train_dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
val_dataset = (val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
test_dataset = (test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [ ]:
class_weights = compute_class_weight(class_weight="balanced",classes=np.unique(train_data["label"]),y=train_data["label"])
class_weights

array([ 4.37305053,  2.78174603,  1.30224782, 12.3633157 ,  1.2855309 ,
        0.21338772, 10.11544012])

In [ ]:
class_weights = dict(enumerate(class_weights))
class_weights

{0: np.float64(4.37305053025577),
 1: np.float64(2.7817460317460316),
 2: np.float64(1.3022478172023035),
 3: np.float64(12.36331569664903),
 4: np.float64(1.285530900421786),
 5: np.float64(0.21338772031292808),
 6: np.float64(10.115440115440116)}

In [20]:
model_results = []

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
num_classes = 7
basic_cnn = models.Sequential([
    layers.Input(shape=(224,224,3)),
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128,activation="relu"),
    layers.Dense(num_classes,activation="softmax")
])


basic_cnn.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 186624)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │    23,888,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,908,295 (91.20 MB)

 Trainable params: 23,908,295 (91.20 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
basic_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [ ]:
history_basic = basic_cnn.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights)

Epoch 1/5


220/220 ━━━━━━━━━━━━━━━━━━━━ 330s 1s/step - accuracy: 0.3786 - loss: 1.8380 - val_accuracy: 0.4075 - val_loss: 1.5757
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 331s 1s/step - accuracy: 0.4478 - loss: 1.5857 - val_accuracy: 0.4754 - val_loss: 1.3882
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 298s 1s/step - accuracy: 0.4762 - loss: 1.4182 - val_accuracy: 0.5133 - val_loss: 1.2243
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 307s 1s/step - accuracy: 0.4886 - loss: 1.3352 - val_accuracy: 0.4647 - val_loss: 1.2837
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 316s 1s/step - accuracy: 0.5051 - loss: 1.2836 - val_accuracy: 0.3429 - val_loss: 1.5566


In [ ]:
basic_cnn.save("../models/basic_cnn.h5")

In [ ]:
test_loss, test_accuracy = basic_cnn.evaluate(test_dataset)
print("Test Accuracy:", test_accuracy)

47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 376ms/step - accuracy: 0.3180 - loss: 1.5563
Test Accuracy: 0.31803059577941895


In [21]:
from tensorflow.keras.models import load_model
basic_load=load_model( "../models/basic_cnn.h5")

W0000 00:00:1785304456.165910  344355 cpu_allocator_impl.cc:82] Allocation of 95551488 exceeds 10% of free system memory.
W0000 00:00:1785304456.229266  344355 cpu_allocator_impl.cc:82] Allocation of 95551488 exceeds 10% of free system memory.
W0000 00:00:1785304456.380616  344355 cpu_allocator_impl.cc:82] Allocation of 95551488 exceeds 10% of free system memory.
W0000 00:00:1785304459.757788  344355 cpu_allocator_impl.cc:82] Allocation of 95551488 exceeds 10% of free system memory.


In [64]:
train_loss,train_acc=basic_load.evaluate(train_dataset)
val_loss,val_acc=basic_load.evaluate(val_dataset)
test_loss,test_acc=basic_load.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 108s 449ms/step - accuracy: 0.3939 - loss: 1.4314
47/47 ━━━━━━━━━━━━━━━━━━━━ 17s 349ms/step - accuracy: 0.3429 - loss: 1.5566
47/47 ━━━━━━━━━━━━━━━━━━━━ 16s 347ms/step - accuracy: 0.3180 - loss: 1.5563


In [ ]:
model_results = []
model_results.append({
    "Model": "Basic CNN",
    "Train Accuracy": 0.3939,
    "Validation Accuracy": 0.3429,
    "Test Accuracy": 0.3180,
    "Train Loss": 1.4314,
    "Validation Loss": 1.5566,
    "Test Loss": 1.5563

})

In [33]:
model_results

[{'Model': 'Basic CNN',
  'Train Accuracy': 0.3939,
  'Validation Accuracy': 0.3429,
  'Test Accuracy': 0.318,
  'Train Loss': 1.4314,
  'Validation Loss': 1.5566,
  'Test Loss': 1.5563}]

In [ ]:

deep_cnn = models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.Conv2D(32,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.Conv2D(64,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.Conv2D(128,(3,3),activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(256,activation="relu"),
    layers.Dense(7,activation="softmax")
])


deep_cnn.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 220, 220, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 110, 110, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 108, 108, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 106, 106, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 53, 53, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 51, 51, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 49, 49, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 24, 24, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 73728)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │    18,874,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,163,431 (73.10 MB)

 Trainable params: 19,163,431 (73.10 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
deep_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [ ]:
history_deep = deep_cnn.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1029s 5s/step - accuracy: 0.2508 - loss: 1.8736 - val_accuracy: 0.3009 - val_loss: 1.6939
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1052s 5s/step - accuracy: 0.4245 - loss: 1.7396 - val_accuracy: 0.2177 - val_loss: 1.9853
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1118s 5s/step - accuracy: 0.4608 - loss: 1.5566 - val_accuracy: 0.1298 - val_loss: 2.1647
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1162s 5s/step - accuracy: 0.4464 - loss: 1.4508 - val_accuracy: 0.2543 - val_loss: 1.8309
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1101s 5s/step - accuracy: 0.4820 - loss: 1.4019 - val_accuracy: 0.4454 - val_loss: 1.3969


In [84]:
deep_cnn.save(
    "../models/deep_cnn.h5"
)

In [85]:
deep_test_loss, deep_test_accuracy = deep_cnn.evaluate(
    test_dataset
)


print(
    "Deep CNN Test Accuracy:",
    deep_test_accuracy
)

47/47 ━━━━━━━━━━━━━━━━━━━━ 50s 1s/step - accuracy: 0.4351 - loss: 1.4103
Deep CNN Test Accuracy: 0.43512973189353943


In [22]:
deep_load=load_model("../models/deep_cnn.h5")

W0000 00:00:1785304474.944874  344355 cpu_allocator_impl.cc:82] Allocation of 75497472 exceeds 10% of free system memory.


In [23]:
train_loss,train_acc=deep_load.evaluate(train_dataset)
val_loss,val_acc=deep_load.evaluate(val_dataset)
test_loss,test_acc=deep_load.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 263s 1s/step - accuracy: 0.4639 - loss: 1.3339
47/47 ━━━━━━━━━━━━━━━━━━━━ 47s 996ms/step - accuracy: 0.4454 - loss: 1.3969
47/47 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - accuracy: 0.4351 - loss: 1.4103


In [ ]:
model_results.append({

    "Model": "Deep CNN",
    "Train Accuracy": 0.4639,
    "Validation Accuracy": 0.4454,
    "Test Accuracy": 0.4351,
    "Train Loss":1.3339,
    "Validation Loss": 1.3969,
    "Test Loss": 1.4103

})

In [ ]:
from tensorflow.keras import layers, models


cnn_batchnorm = models.Sequential([
    layers.Input(shape=(224,224,3)),
    # Block 1
    layers.Conv2D(32,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(32,(3,3),adding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 2
    layers.Conv2D(64,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(64,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    # Block 3
    layers.Conv2D(128,(3,3),padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPooling2D((2,2)),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dense(7,activation="softmax")

])


cnn_batchnorm.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 224, 224, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 112, 112, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_10 (MaxPooling2D) │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_11 (MaxPooling2D) │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 158,119 (617.65 KB)

 Trainable params: 157,479 (615.15 KB)

 Non-trainable params: 640 (2.50 KB)

In [ ]:
cnn_batchnorm.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [ ]:
history_batchnorm = cnn_batchnorm.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1303s 6s/step - accuracy: 0.3827 - loss: 1.6942 - val_accuracy: 0.0113 - val_loss: 2.5723
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1446s 7s/step - accuracy: 0.4615 - loss: 1.5013 - val_accuracy: 0.0746 - val_loss: 2.6365
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1450s 7s/step - accuracy: 0.4785 - loss: 1.4197 - val_accuracy: 0.3509 - val_loss: 1.7836
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1335s 6s/step - accuracy: 0.4909 - loss: 1.3708 - val_accuracy: 0.3715 - val_loss: 1.7862
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1419s 6s/step - accuracy: 0.4919 - loss: 1.3380 - val_accuracy: 0.5100 - val_loss: 1.3453


In [90]:
cnn_batchnorm.save(
    "../models/cnn_batchnorm.h5"
)

In [91]:
batchnorm_loss, batchnorm_accuracy = cnn_batchnorm.evaluate(
    test_dataset
)


print(
    "CNN BatchNorm Test Accuracy:",
    batchnorm_accuracy
)

47/47 ━━━━━━━━━━━━━━━━━━━━ 48s 1s/step - accuracy: 0.4671 - loss: 1.3744
CNN BatchNorm Test Accuracy: 0.46706587076187134


In [61]:
norm_load=load_model("../models/cnn_batchnorm.h5")

In [62]:
train_loss,train_acc=norm_load.evaluate(train_dataset)
val_loss,val_acc=norm_load.evaluate(val_dataset)
test_loss,test_acc=norm_load.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 304s 1s/step - accuracy: 0.5208 - loss: 1.2281
47/47 ━━━━━━━━━━━━━━━━━━━━ 58s 1s/step - accuracy: 0.5100 - loss: 1.3453
47/47 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step - accuracy: 0.4671 - loss: 1.3744


In [ ]:
model_results.append({

    "Model": "CNN + BatchNormalization",
    "Train Accuracy": 0.5208,
    "Validation Accuracy": 0.5100,
    "Test Accuracy": 0.4671,
    "Train Loss": 1.2281,
    "Validation Loss": 1.3453,
    "Test Loss": 1.3744

})

In [36]:
model_results

[{'Model': 'Basic CNN',
  'Train Accuracy': 0.3939,
  'Validation Accuracy': 0.3429,
  'Test Accuracy': 0.318,
  'Train Loss': 1.4314,
  'Validation Loss': 1.5566,
  'Test Loss': 1.5563},
 {'Model': 'Deep CNN',
  'Train Accuracy': 0.4639,
  'Validation Accuracy': 4454,
  'Test Accuracy': 0.4351,
  'Train Loss': 1.3339,
  'Validation Loss': 1.3969,
  'Test Loss': 1.4103},
 {'Model': 'CNN + BatchNormalization',
  'Train Accuracy': 0.5208,
  'Validation Accuracy': 0.51,
  'Test Accuracy': 0.4671,
  'Train Loss': 1.2281,
  'Validation Loss': 1.3453,
  'Test Loss': 1.3744}]

In [93]:
from tensorflow.keras import layers, models

cnn_dropout = models.Sequential([

    layers.Input(shape=(224, 224, 3)),

    # Block 1
    layers.Conv2D(32, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 2
    layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    # Block 3
    layers.Conv2D(128, (3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),
    layers.Dropout(0.25),

    layers.Flatten(),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),

    layers.Dense(7, activation="softmax")

])

cnn_dropout.summary()

Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_17 (Conv2D)              │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_18 (Conv2D)              │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_19 (Conv2D)              │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 100352)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │    25,690,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,785,415 (98.36 MB)

 Trainable params: 25,785,415 (98.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
cnn_dropout.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [ ]:
history_dropout = cnn_dropout.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 453s 2s/step - accuracy: 0.3578 - loss: 2.0209 - val_accuracy: 0.5040 - val_loss: 1.8357
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 494s 2s/step - accuracy: 0.4382 - loss: 1.8607 - val_accuracy: 0.3209 - val_loss: 1.7720
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 496s 2s/step - accuracy: 0.4633 - loss: 1.8122 - val_accuracy: 0.3995 - val_loss: 1.7665
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 475s 2s/step - accuracy: 0.4292 - loss: 1.7730 - val_accuracy: 0.4461 - val_loss: 1.6570
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 428s 2s/step - accuracy: 0.4355 - loss: 1.6989 - val_accuracy: 0.4960 - val_loss: 1.4071


In [99]:
cnn_dropout.save("../models/cnn_dropout.h5")

In [100]:
dropout_loss, dropout_accuracy = cnn_dropout.evaluate(test_dataset)

print("Test Accuracy:", dropout_accuracy)

47/47 ━━━━━━━━━━━━━━━━━━━━ 20s 423ms/step - accuracy: 0.4904 - loss: 1.4353
Test Accuracy: 0.4903526306152344


In [59]:
drop_load=load_model("../models/cnn_dropout.h5")

In [60]:
train_loss,train_acc=drop_load.evaluate(train_dataset)
val_loss,val_acc=drop_load.evaluate(val_dataset)
test_loss,test_acc=drop_load.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 138s 592ms/step - accuracy: 0.5001 - loss: 1.3619
47/47 ━━━━━━━━━━━━━━━━━━━━ 28s 579ms/step - accuracy: 0.4960 - loss: 1.4071
47/47 ━━━━━━━━━━━━━━━━━━━━ 28s 580ms/step - accuracy: 0.4904 - loss: 1.4353


In [ ]:
model_results.append({

    "Model": "CNN Dropout",
    "Train Accuracy": 0.5001,
    "Validation Accuracy": 0.4960,
    "Test Accuracy": 0.4904,
    "Train Loss": 1.3619,
    "Validation Loss": 1.4071,
    "Test Loss": 1.4353

})

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

mobilenet_model = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

mobilenet_model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,855 (9.24 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [ ]:
history_mobilenet = mobilenet_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=5,
    class_weight=class_weights

)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 212s 924ms/step - accuracy: 0.2797 - loss: 1.9692 - val_accuracy: 0.3189 - val_loss: 1.6758
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 198s 868ms/step - accuracy: 0.3967 - loss: 1.6352 - val_accuracy: 0.5173 - val_loss: 1.3824
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 225s 990ms/step - accuracy: 0.4388 - loss: 1.5421 - val_accuracy: 0.5060 - val_loss: 1.3622
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 209s 919ms/step - accuracy: 0.4700 - loss: 1.4631 - val_accuracy: 0.5473 - val_loss: 1.2628
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 196s 862ms/step - accuracy: 0.4940 - loss: 1.4211 - val_accuracy: 0.5306 - val_loss: 1.2496


In [104]:
mobilenet_model.save(
    "../models/mobilenetv2.h5"
)

In [105]:
mobilenet_loss, mobilenet_accuracy = mobilenet_model.evaluate(
    test_dataset
)

print("MobileNetV2 Test Accuracy:", mobilenet_accuracy)

47/47 ━━━━━━━━━━━━━━━━━━━━ 34s 711ms/step - accuracy: 0.5030 - loss: 1.3032
MobileNetV2 Test Accuracy: 0.5029940009117126


In [57]:
mobile_load=load_model("../models/mobilenetv2.h5")

In [58]:
train_loss,train_acc=mobile_load.evaluate(train_dataset)
val_loss,val_acc=mobile_load.evaluate(val_dataset)
test_loss,test_acc=mobile_load.evaluate(test_dataset)

220/220 ━━━━━━━━━━━━━━━━━━━━ 228s 989ms/step - accuracy: 0.5515 - loss: 1.2364
47/47 ━━━━━━━━━━━━━━━━━━━━ 39s 830ms/step - accuracy: 0.5306 - loss: 1.2496
47/47 ━━━━━━━━━━━━━━━━━━━━ 39s 834ms/step - accuracy: 0.5030 - loss: 1.3032


In [ ]:
model_results.append({

    "Model": " MobileNet",
    "Train Accuracy": 0.5515,
    "Validation Accuracy":0.5306,
    "Test Accuracy": 0.5038,
    "Train Loss": 1.2364,
    "Validation Loss": 1.2496,
    "Test Loss": 1.3032

})

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras import layers, models

base_model = DenseNet121(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
densenet121_model = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])

densenet121_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet121 (Functional)        │ (None, 7, 7, 1024)     │     7,037,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,169,607 (27.35 MB)

 Trainable params: 132,103 (516.03 KB)

 Non-trainable params: 7,037,504 (26.85 MB)

In [ ]:
densenet121_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [ ]:
history_densenet = densenet121_model.fit(train_dataset,validation_data=val_dataset,epochs=5,class_weight=class_weights)

Epoch 1/5


/home/aximsoft/snap/code/253/.local/share/virtualenvs/SkinCancer_Disease-Fur3-H-s/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


220/220 ━━━━━━━━━━━━━━━━━━━━ 825s 4s/step - accuracy: 0.1608 - loss: 2.0152 - val_accuracy: 0.2337 - val_loss: 1.7907
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 798s 4s/step - accuracy: 0.2458 - loss: 1.7291 - val_accuracy: 0.4015 - val_loss: 1.6136
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 768s 3s/step - accuracy: 0.3213 - loss: 1.6130 - val_accuracy: 0.4947 - val_loss: 1.4447
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 659s 3s/step - accuracy: 0.3854 - loss: 1.5616 - val_accuracy: 0.5453 - val_loss: 1.3308
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 769s 3s/step - accuracy: 0.4170 - loss: 1.5025 - val_accuracy: 0.5606 - val_loss: 1.2806


In [ ]:
densenet121_model.save("../models/densenet121.h5")

In [ ]:
densenet121_loss, densenet121_accuracy = densenet121_model.evaluate(test_dataset)

print("DenseNet121 Test Accuracy:",densenet121_accuracy)

47/47 ━━━━━━━━━━━━━━━━━━━━ 137s 3s/step - accuracy: 0.5589 - loss: 1.3051
DenseNet121 Test Accuracy: 0.5588822364807129


In [55]:
from tensorflow.keras.models import load_model
desnet_load=load_model( "../models/densenet121.h5")

In [56]:
train_loss,train_acc=desnet_load.evaluate(train_dataset)
val_loss,val_acc=desnet_load.evaluate(val_dataset)
test_loss,test_acc=desnet_load.evaluate(test_dataset)

I0000 00:00:1785298636.245640  206263 shuffle_dataset_op.cc:453] ShuffleDatasetV3:7: Filling up shuffle buffer (this may take a while): 448 of 1000
I0000 00:00:1785298647.422235  206263 shuffle_dataset_op.cc:483] Shuffle buffer filled.


220/220 ━━━━━━━━━━━━━━━━━━━━ 1159s 5s/step - accuracy: 0.5417 - loss: 1.3811
47/47 ━━━━━━━━━━━━━━━━━━━━ 243s 5s/step - accuracy: 0.5606 - loss: 1.2806
47/47 ━━━━━━━━━━━━━━━━━━━━ 141s 3s/step - accuracy: 0.5589 - loss: 1.3051


In [ ]:
model_results.append({

    "Model": "DesNet",
    "Train Accuracy": 0.5417,
    "Validation Accuracy": 0.5606,
    "Test Accuracy": 0.5589,
    "Train Loss": 1.3811,
    "Validation Loss": 1.2806,
    "Test Loss": 1.3051

})

In [37]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications.efficientnet import preprocess_input

In [ ]:
augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
    layers.RandomBrightness(0.1)
])

In [ ]:
def preprocess_image_efficientnet(image_path):

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(224,224))
    image = image.astype(np.float32)
    image = preprocess_input(image)
    return image

In [ ]:
def create_efficientnet_dataset(data, training=True):

    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    def load_image(path, label):
        image = tf.numpy_function(preprocess_image_efficientnet,[path],tf.float32)
        image.set_shape((224,224,3))
        if training:
            image = augmentation(image)
        return image, label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [ ]:
train_eff = create_efficientnet_dataset(train_data,training=True)
val_eff = create_efficientnet_dataset(val_data,training=False)
test_eff = create_efficientnet_dataset(test_data,training=False)

In [ ]:
train_eff=(train_eff.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE))

val_eff = (val_eff.batch(32).prefetch(tf.data.AUTOTUNE))

test_eff = (
    test_eff
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0


base_model = EfficientNetB0(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False
efficientnet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")])
efficientnet_model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,214,442 (16.08 MB)

 Trainable params: 164,871 (644.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [ ]:
efficientnet_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                           loss="sparse_categorical_crossentropy",
                           metrics=["accuracy"])

In [ ]:
history_eff_aug = efficientnet_model.fit(train_eff,validation_data=val_eff,epochs=5,class_weight=class_weights)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 328s 1s/step - accuracy: 0.2427 - loss: 1.9366 - val_accuracy: 0.4754 - val_loss: 1.6120
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 341s 2s/step - accuracy: 0.3715 - loss: 1.7045 - val_accuracy: 0.5326 - val_loss: 1.4432
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 374s 2s/step - accuracy: 0.4391 - loss: 1.5373 - val_accuracy: 0.5466 - val_loss: 1.3222
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 438s 2s/step - accuracy: 0.4762 - loss: 1.4452 - val_accuracy: 0.5699 - val_loss: 1.2167
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 434s 2s/step - accuracy: 0.4845 - loss: 1.3847 - val_accuracy: 0.5932 - val_loss: 1.1554


In [46]:
efficientnet_model.save("../models/efficientnetb0.h5")

In [ ]:
eff_aug_loss, eff_aug_accuracy = efficientnet_model.evaluate(test_eff)

print(
    "EfficientNetB0 + Augmentation Accuracy:",
    eff_aug_accuracy
)

47/47 ━━━━━━━━━━━━━━━━━━━━ 73s 2s/step - accuracy: 0.5875 - loss: 1.1939
EfficientNetB0 + Augmentation Accuracy: 0.5874916911125183


In [ ]:
from tensorflow.keras.models import load_model
eff_load=load_model( "../models/efficientnetb0.h5")

In [51]:
train_loss,train_acc=eff_load.evaluate(train_eff)
val_loss,val_acc=eff_load.evaluate(val_eff)
test_loss,test_acc=eff_load.evaluate(test_eff)

I0000 00:00:1785295277.946485   96032 shuffle_dataset_op.cc:453] ShuffleDatasetV3:23: Filling up shuffle buffer (this may take a while): 682 of 1000
I0000 00:00:1785295282.450640   96032 shuffle_dataset_op.cc:483] Shuffle buffer filled.


220/220 ━━━━━━━━━━━━━━━━━━━━ 669s 3s/step - accuracy: 0.5835 - loss: 1.3074
47/47 ━━━━━━━━━━━━━━━━━━━━ 122s 3s/step - accuracy: 0.5992 - loss: 1.2157
47/47 ━━━━━━━━━━━━━━━━━━━━ 137s 3s/step - accuracy: 0.5902 - loss: 1.2472


In [ ]:
model_results.append({

    "Model": "EfficientNet",
    "Train Accuracy": 0.5835,
    "Validation Accuracy": 0.5992,
    "Test Accuracy": 0.5902,
    "Train Loss": 1.3074,
    "Validation Loss": 1.2157,
    "Test Loss": 1.2472

})

In [32]:
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
def preprocess_image_resnet(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image = cv2.resize(image,(224,224))
    image = image.astype(np.float32)
    image = preprocess_input(image)
    return image

In [ ]:
def create_resnet_dataset(data, training=True):

    image_paths = data["image_path"].values
    labels = data["label"].values
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    def load_image(path, label):
        image = tf.numpy_function(preprocess_image_resnet,[path],tf.float32)
        image.set_shape((224,224,3))
        if training:
            image = augmentation(image)
        return image, label
    dataset = dataset.map(load_image,num_parallel_calls=tf.data.AUTOTUNE)
    return dataset

In [ ]:
train_resnet = create_resnet_dataset(train_data,training=True)
val_resnet = create_resnet_dataset(val_data,training=False)
test_resnet = create_resnet_dataset(test_data,training=False)

In [ ]:
train_resnet = (train_resnet.shuffle(1000).batch(32).prefetch(tf.data.AUTOTUNE))
val_resnet = (val_resnet.batch(32).prefetch(tf.data.AUTOTUNE))
test_resnet = (test_resnet.batch(32).prefetch(tf.data.AUTOTUNE))

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models


base_model = ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_model.trainable = False

resnet50_model = models.Sequential([

    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(7,activation="softmax")

])


resnet50_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,850,887 (90.98 MB)

 Trainable params: 263,175 (1.00 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [ ]:
resnet50_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

In [ ]:
history_resnet_aug=resnet50_model.fit(train_resnet,validation_data=val_resnet,epochs=5,class_weight=class_weights)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 689s 3s/step - accuracy: 0.3133 - loss: 2.0045 - val_accuracy: 0.3562 - val_loss: 1.7913
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 689s 3s/step - accuracy: 0.4496 - loss: 1.5953 - val_accuracy: 0.2976 - val_loss: 1.8995
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 1197s 5s/step - accuracy: 0.4612 - loss: 1.4477 - val_accuracy: 0.3609 - val_loss: 1.7524
Epoch 4/5


I0000 00:00:1785259599.817074  312763 shuffle_dataset_op.cc:453] ShuffleDatasetV3:52: Filling up shuffle buffer (this may take a while): 694 of 1000
I0000 00:00:1785259604.579847  312763 shuffle_dataset_op.cc:483] Shuffle buffer filled.


220/220 ━━━━━━━━━━━━━━━━━━━━ 1436s 6s/step - accuracy: 0.4815 - loss: 1.3882 - val_accuracy: 0.4141 - val_loss: 1.6267
Epoch 5/5


I0000 00:00:1785261035.278114  342601 shuffle_dataset_op.cc:453] ShuffleDatasetV3:52: Filling up shuffle buffer (this may take a while): 683 of 1000
I0000 00:00:1785261039.832370  342601 shuffle_dataset_op.cc:483] Shuffle buffer filled.


220/220 ━━━━━━━━━━━━━━━━━━━━ 1425s 6s/step - accuracy: 0.5188 - loss: 1.3137 - val_accuracy: 0.4314 - val_loss: 1.6008


In [ ]:
resnet50_model.save("../models/resnet50.h5")

In [ ]:
resnet50_loss, resnet50_accuracy = resnet50_model.evaluate(test_resnet)
print("ResNet50 Test Accuracy:",resnet50_accuracy)

47/47 ━━━━━━━━━━━━━━━━━━━━ 254s 5s/step - accuracy: 0.4225 - loss: 1.6381
ResNet50 Test Accuracy: 0.42248836159706116


In [48]:
from tensorflow.keras.models import load_model
resnet_load=load_model( "../models/resnet50.h5")

In [52]:
train_loss,train_acc=resnet_load.evaluate(train_resnet)
val_loss,val_acc=resnet_load.evaluate(val_resnet)
test_loss,test_acc=resnet_load.evaluate(test_resnet)

I0000 00:00:1785296192.250984  130448 shuffle_dataset_op.cc:453] ShuffleDatasetV3:42: Filling up shuffle buffer (this may take a while): 433 of 1000
I0000 00:00:1785296204.243048  130448 shuffle_dataset_op.cc:483] Shuffle buffer filled.


220/220 ━━━━━━━━━━━━━━━━━━━━ 1264s 6s/step - accuracy: 0.2586 - loss: 1.6925
47/47 ━━━━━━━━━━━━━━━━━━━━ 495s 11s/step - accuracy: 0.3049 - loss: 1.6161
47/47 ━━━━━━━━━━━━━━━━━━━━ 282s 6s/step - accuracy: 0.3107 - loss: 1.6175


In [ ]:
model_results.append({

    "Model": "ResNet",
    "Train Accuracy": 0.2586,
    "Validation Accuracy": 0.3049,
    "Test Accuracy": 0.3107,
    "Train Loss": 1.6925,
    "Validation Loss": 1.6161,
    "Test Loss": 1.6175

})

In [57]:
print(model_results)

[{'Model': 'Basic CNN', 'Train Accuracy': 0.3939, 'Validation Accuracy': 0.3429, 'Test Accuracy': 0.318, 'Train Loss': 1.4314, 'Validation Loss': 1.5566, 'Test Loss': 1.5563}, {'Model': 'Deep CNN', 'Train Accuracy': 0.4639, 'Validation Accuracy': 4454, 'Test Accuracy': 0.4351, 'Train Loss': 1.3339, 'Validation Loss': 1.3969, 'Test Loss': 1.4103}, {'Model': 'CNN + BatchNormalization', 'Train Accuracy': 0.5208, 'Validation Accuracy': 0.51, 'Test Accuracy': 0.4671, 'Train Loss': 1.2281, 'Validation Loss': 1.3453, 'Test Loss': 1.3744}, {'Model': 'CNN Dropout', 'Train Accuracy': 0.5001, 'Validation Accuracy': 0.496, 'Test Accuracy': 0.4904, 'Train Loss': 1.3619, 'Validation Loss': 1.4071, 'Test Loss': 1.4353}, {'Model': ' MobileNet', 'Train Accuracy': 0.5515, 'Validation Accuracy': 0.5306, 'Test Accuracy': 0.5038, 'Train Loss': 1.2364, 'Validation Loss': 1.2496, 'Test Loss': 1.3032}, {'Model': 'DesNet', 'Train Accuracy': 0.5417, 'Validation Accuracy': 0.5606, 'Test Accuracy': 0.5589, 'Train

In [68]:
results=pd.DataFrame(model_results)

In [69]:
results

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Train Loss,Validation Loss,Test Loss
0,Basic CNN,0.3939,0.3429,0.3180,1.4314,1.5566,1.5563
1,Deep CNN,0.4639,0.4454,0.4351,1.3339,1.3969,1.4103
2,CNN + BatchNormalization,0.5208,0.5100,0.4671,1.2281,1.3453,1.3744
3,CNN Dropout,0.5001,0.4960,0.4904,1.3619,1.4071,1.4353
4,MobileNet,0.5515,0.5306,0.5038,1.2364,1.2496,1.3032
5,DesNet,0.5417,0.5606,0.5589,1.3811,1.2806,1.3051
6,EfficientNet,0.5835,0.5992,0.5902,1.3074,1.2157,1.2472
7,ResNet,0.2586,0.3049,0.3107,1.6925,1.6161,1.6175


In [70]:
results.to_csv("../resutls.csv")